In [1]:
# Add Direction and Current from U/V on current meters

In [6]:
import xarray as xr
import numpy as np
import EcoFOCIpy.io.erddap as erddap

import EcoFOCIpy.metaconfig.load_config as load_config

In [7]:
sample_data_dir = '/Users/bell/Programs/EcoFOCIpy/'
user_data_dir = '/Users/bell/ecoraid/2024/Moorings/24kum2a/final_data_cf/'
filename = '24kum2a_an9_0005m.nc'

### Post Manual QC load data and rebuild nc file

- using excel for a few points or a dynamic web map for multiple (a tool on ecofoci-field.pmel.noaa.gov) or any other method to identify spikes
- remove values above 32psu and below 30psu

**NOTE** if you use excel, be cautious about how your time is formatted (its important) - use custom formatting and make it of the form `yyyy-mm-ddTHH:MM:SSZ` to assure proper read in to xarray

*TODO:* use erddap hosted data 

In [8]:
# this loads the initial working netcdf file
wop_nc_qc = xr.load_dataset(user_data_dir+filename)

In [9]:
wop_nc_qc

<xarray.Dataset> Size: 198kB
Dimensions:         (time: 2743, depth: 1, latitude: 1, longitude: 1)
Coordinates:
  * time            (time) datetime64[ns] 22kB 2024-05-16T18:00:00 ... 2024-0...
  * depth           (depth) float64 8B 5.0
  * latitude        (latitude) float64 8B 58.26
  * longitude       (longitude) float64 8B -163.6
Data variables:
    current_speed   (time, depth, latitude, longitude) float64 22kB 65.99 ......
    u_curr_comp     (time, depth, latitude, longitude) float64 22kB 62.27 ......
    v_curr_comp     (time, depth, latitude, longitude) float64 22kB -21.84 .....
    oxy_conc        (time, depth, latitude, longitude) float64 22kB 366.6 ......
    oxy_percentsat  (time, depth, latitude, longitude) float64 22kB 103.6 ......
    temperature     (time, depth, latitude, longitude) float64 22kB 0.4841 .....
    pressure        (time, depth, latitude, longitude) float64 22kB nan ... nan
    oxy_conc_umkg   (time, depth, latitude, longitude) float64 22kB 357.6 ......
Attributes: (12/33)
    Latitude_DegMMddW:                      58 15.713 N
    Longitude_DegMMddN:                     163 33.441 W
    MooringID:                              24KUM-2A
    platform_deployment_date:               2024-05-16T17:49:00Z
    platform_deployment_cruise_name:        DY2406
    platform_recovery_date:                 2024-09-08T00:00:00Z
    ...                                     ...
    InstrumentSerialNumber:                 858
    InstrumentType:                         RCM9
    date_created:                           2024-12-26T07:35:00Z
    date_modified:                          
    QC_indicator:                           unknown
    history:                                Trimmed to deployment.\nTrimmed t...

In [10]:
sample_data_dir = '/Users/bell/Programs/EcoFOCIpy/'
inst_meta_file = sample_data_dir+'staticdata/instr_metaconfig/rcm_cf.yaml'

In [14]:
def calculate_ocean_current(u, v):
    """
    Calculates current speed and direction (oceanographic convention).
    
    Parameters:
    u : Zonal velocity (Eastward positive)
    v : Meridional velocity (Northward positive)
    
    Returns:
    speed : Magnitude of the current
    direction : Degrees clockwise from North (0-360)
    """
    # 1. Calculate Speed (Magnitude)
    # Equivalent to sqrt(u^2 + v^2)
    speed = np.hypot(u, v)
    
    # 2. Calculate Direction (Oceanographic convention: "To")
    # Using atan2(u, v) automatically handles the quadrants and 
    # aligns 0 degrees with the positive v-axis (North).
    direction_rad = np.arctan2(u, v)
    direction_deg = np.degrees(direction_rad)
    
    # 3. Normalize to 0-360 degrees
    direction_deg = (direction_deg + 360) % 360
    
    return speed, direction_deg

In [20]:
wop_nc_qc['current_direction'] = None

In [21]:
wop_nc_qc['current_direction'].data = calculate_ocean_current(wop_nc_qc['u_curr_comp'],wop_nc_qc['v_curr_comp'])